In [ ]:
"""
Authors: Shanmuga Venkatachalam, Harideep Nair, Shreyas Chaudhari
"""

### Importing libraries ###

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import save_image

import numpy as np
import matplotlib.pyplot as plt
import time
import math



### Rectangular Cropping ###

    # Crops a tensor into a given rectangular shape starting from a given position.

    # Args: pos         - Top left co-ordinates of the crop position. Could be a scalar or tuple.
    #       size        - Rectangular crop size. Could be a scalar or tuple.
    #       tensor      - Input tensor of any shape. Typical shape is [in_channels,height,width].
    # Returns a corresponding cropped tensor with the last two dimensions of its shape equal to crop size

class Crop(object):
    def __init__(self, pos, size):
        if not isinstance(pos,tuple):
            self.pos  = (pos,pos)
        else:
            self.pos  = pos
        if not isinstance(size,tuple):
            self.size  = (size,size)
        else:
            self.size  = size

    def __call__(self, tensor):
        return tensor[:,self.pos[0]:self.pos[0]+self.size[0],self.pos[1]:self.pos[1]+self.size[1]]



### On-Off filtering ###

    # Performs On and Off filtering on a given tensor as a preliminary edge-detection mechanism.
    # On filter  - Takes a pixel value and subtracts the mean of its surrounding pixels from it.
    # Off filter - Subtracts a pixel value from the mean of its surrounding pixels.

    # Args: window_size - Scalar window size of the filter. Filter's shape is assumed to be a square.
    #       tensor      - Input tensor of shape [in_channels,height,width].
    # Returns concatenated On and Off versions of the input tensor with output shape [in_channels,2*height,width].

class OnOff(object):
    def __init__(self, window_size=3):
        self.window_size = window_size

    def __call__(self, tensor):
        tensor                 = torch.unsqueeze(tensor, 0)
        mid                    = int((self.window_size-1)/2)

        onfilter               = -1*torch.ones((tensor.shape[1],1,self.window_size, self.window_size))
        onfilter[0,:,mid,mid]  = (self.window_size**2)-1

        offfilter = torch.ones((tensor.shape[1],1,self.window_size, self.window_size))
        offfilter[0,:,mid,mid] = 1-(self.window_size**2)

        on                     = F.conv2d(tensor, onfilter, stride=1,\
                                          padding=((self.window_size-1)//2,(self.window_size-1)//2), groups=tensor.shape[1])
        off                    = F.conv2d(tensor, offfilter, stride=1,\
                                          padding=((self.window_size-1)//2,(self.window_size-1)//2), groups=tensor.shape[1])
        out                    = torch.cat([on.squeeze_(0),off.squeeze_(0)], dim=1)
        out[out<0]             = 0

        return out



### Pos-Neg filtering ###

    # Performs Positive and Negative filtering on a given tensor.
    # The input tensor is binarized first and then it's negative is appended.
    # This ensures all RFs have equal number of spikes.

    # Args: window_size - Scalar window size of the filter. Filter's shape is assumed to be a square.
    #       tensor      - Input tensor of shape [in_channels,height,width].
    # Returns concatenated On and Off versions of the input tensor with output shape [in_channels,2*height,width].

class PosNeg_SpikeEvents(object):
    def __init__(self, pn_threshold):
        self.pn_thresh = pn_threshold

    def __call__(self, tensor):
        maxt                                    = torch.max(tensor)
        tensor[tensor < (self.pn_thresh*maxt)]  = 0
        tensor[tensor >= (self.pn_thresh*maxt)] = 1
        tensor_neg                              = tensor.clone()
        tensor_neg[tensor_neg == 0]             = 2
        tensor_neg[tensor_neg == 1]             = 0
        tensor_neg[tensor_neg == 2]             = 1

        out                    = torch.cat([tensor, tensor_neg], dim=0)

        return out

class PosNeg_SpikeTimes(object):
    def __init__(self, pn_threshold):
        self.pn_thresh = pn_threshold

    def __call__(self, tensor):
        maxt                                    = torch.max(tensor)
        tensor[tensor >= (self.pn_thresh*maxt)] = maxt
        tensor[tensor < (self.pn_thresh*maxt)]  = 0

        tensor                                  = tensor - maxt
        tensor[tensor == -maxt]                 = float('Inf')
        tensor_neg                              = tensor.clone()
        tensor_neg[tensor_neg == 0]             = 1
        tensor_neg[tensor_neg == float('Inf')]  = 0
        tensor_neg[tensor_neg == 1]             = float('Inf')

        out                    = torch.cat([tensor,tensor_neg], dim=0)

        return out



### On-Off filtering with channel creation ###

    # Performs On and Off filtering on a given tensor as a preliminary edge-detection mechanism.
    # On filter  - Takes a pixel value and subtracts the mean of its surrounding pixels from it.
    # Off filter - Subtracts a pixel value from the mean of its surrounding pixels.

    # Args: window_size - Scalar window size of the filter. Filter's shape is assumed to be a square.
    #       tensor      - Input tensor of shape [in_channels,height,width].
    # Returns On and Off versions of the input tensor as two separate channels with output shape [in_channels*2,height,width].

class OnOff_Channels(object):
    def __init__(self, window_size=3):
        self.window_size = window_size

    def __call__(self, tensor):
        tensor                 = torch.unsqueeze(tensor, 0)
        mid                    = int((self.window_size-1)/2)

        onfilter               = -1*torch.ones((tensor.shape[1],1,self.window_size, self.window_size))
        onfilter[0,:,mid,mid]  = (self.window_size**2)-1

        offfilter = torch.ones((tensor.shape[1],1,self.window_size, self.window_size))
        offfilter[0,:,mid,mid] = 1-(self.window_size**2)

        on                     = F.conv2d(tensor, onfilter, stride=1,\
                                          padding=((self.window_size-1)//2,(self.window_size-1)//2), groups=tensor.shape[1])
        off                    = F.conv2d(tensor, offfilter, stride=1,\
                                          padding=((self.window_size-1)//2,(self.window_size-1)//2), groups=tensor.shape[1])
        out                    = torch.cat([on.squeeze_(0),off.squeeze_(0)], dim=0)
        out[out<0]             = 0

        return out



### Black-and-white temporal encoding ###

    # Converts a given tensor to black-and-white, and generates its corresponding temporally encoded spiketimes.
    # "Black" is given a spiketime of 'infinity' (no spike), whereas "white" corresponds to a spike at time '0'.

    # Args: threshold   - Any value equal to or higher than the threshold is considered "white"; else "black".
    #       tensor      - Input tensor of any shape. Typical shape is [in_channels,height,width].
    # Returns a corresponding black-and-white tensor of spiketimes with the same shape as input tensor.

class BlackAndWhite(object):
    def __init__(self, threshold = 0.5):
        self.threshold = threshold

    def __call__(self, tensor):
        maxt   = torch.max(tensor)
        tensor[tensor >= (self.threshold*maxt)] = maxt
        tensor[tensor < (self.threshold*maxt)]  = 0
        tensor = tensor - maxt
        tensor[tensor==-maxt] = float('Inf')
        return tensor



### Grayscale temporal encoding ###

    # Generates temporally encoded spiketimes from a given grayscale tensor.
    # Higher the input value, earlier the corresponding spiketime.

    # Args: timeres     - Bit resolution for the output spiketime window. Relates to maximum output spiketime.
    #       threshold   - (optional) Any spiketime equal to or higher than the threshold is considered as "no spike".
    #       tensor      - Input tensor of any shape. Typical shape is [in_channels,height,width].
    # Returns a corresponding tensor of spiketimes with the same shape as input tensor.

class GrayScale(object):
    def __init__(self, timeres, threshold):
        self.time      = (2**timeres)-1
        self.threshold = threshold

    def __call__(self, tensor):
        maxt   = torch.max(tensor)
        tensor = torch.round((tensor/maxt)*self.time)
        tensor = self.time-tensor
        tensor[tensor>=self.threshold] = float('Inf')
        return tensor



### Grayscale temporal encoding with PosNeg Channels ###

    # Generates temporally encoded spiketimes from a given grayscale tensor with positive and negative channels.

    # Args: timeres     - Bit resolution for the output spiketime window. Relates to maximum output spiketime.
    #       tensor      - Input tensor of any shape. Typical shape is [in_channels,height,width].
    # Returns a corresponding tensor of spiketimes with the same shape as input tensor.

class PosNegGrayScale(object):
    def __init__(self, timeres):
        self.tmax = 2**timeres - 1

    def __call__(self, tensor):
        maxval                                  = torch.max(tensor)
        tensor_pos                              = torch.round((tensor/maxval) * self.tmax)
        tensor_neg                              = self.tmax - tensor_pos

        tensor_pos[tensor_pos == self.tmax]     = float('Inf')
        tensor_neg[tensor_neg == self.tmax]     = float('Inf')

        out                                     = torch.cat([tensor_pos,tensor_neg], dim=0)

        return out


### Gaussian encoder used for UCR time-series benchmarks ###
class PopulationCoder:

    """Gaussian receptive field population encoder

    Attributes:
        tmax    (int): Exclusive upper bound on maximum spike time
        n       (int): Number of Gaussian curves to use for each time series sample
        data    (tensor): The data that is to be encoded
        centers (tensor): Tensor of the means of each Gaussian curve
        widths  (tensor): Tensor of the widths of each Gaussian curve
        beta    (int): Common scaling factor on the width of each Gaussian curve
        scaler  (object): Used to scale incoming signals to zero mean, unit variance
    """

    def __init__(self, data, n, tmax, beta=1):
        self.tmax = tmax
        self.n = n
        self.beta = beta
        data_len = data.shape[1]

        # Save scaling parameters
        self.scaler = StandardScaler()
        data = torch.from_numpy(self.scaler.fit_transform(data))


        min_vals, _ = data.min(dim=0)
        max_vals, _ = data.max(dim=0)
        self.widths = beta*torch.abs(max_vals - min_vals) / (n - 2)
        self.widths = self.widths.unsqueeze(-1).repeat(1, self.n)  # --> shape = (data_len, n)

        self.centers = torch.zeros(data_len, n)
        for i in range(data_len):

            idx_vec = torch.arange(n)
            self.centers[i] = (
                min_vals[i] + ((2 * idx_vec - 3)/ 2) * self.widths[i]
            )


    def encode(self, tensor):

        """ Maps data tensor to spike times.

        Args:
            (tensor): The time series signal of shape (data_len,) where data_len is the
                        number of timesteps in one signal

        Returns:
            (tensor): The spike times where float('Inf') indicates not spike. The result has
                        shape (data_len, n)

        """

        tensor = self.scaler.transform(tensor.unsqueeze(0)).squeeze()
        tensor = torch.from_numpy(tensor)

        tensor = tensor.unsqueeze(-1).repeat(1, self.n)
        exc = torch.exp(
            -0.5
            * ((tensor - self.centers) / self.widths)
            * ((tensor - self.centers) / self.widths)
        )
        spike_times = torch.floor((self.tmax + 1) * (1 - exc))
        spike_times[spike_times >= self.tmax] = float("Inf")

        return spike_times



### Converts spiketimes to a tensor over time with a separate temporal dimension ###

    # Adds a first dimension for time.
    # The entry for a spiketime, say time 't', will have zeros until time 't-1' and ones thereafter.

    # Args: spiketimes - Input spiketimes with time values.
    #       time       - Length of the time window to be modeled.
    # Returns a corresponding temporal tensor of spiketimes with an additional 'time' dimension.

def AddTemporalDim(spiketimes, time):
    spiketimes                    = spiketimes.permute(1,2,0)
    temprows, tempcols, tempnprev = spiketimes.shape
    const                         = torch.arange(time).repeat(temprows,tempcols,tempnprev,1).permute(3,0,1,2)

    spikes                        = spiketimes.unsqueeze(0).repeat(time,1,1,1)
    spikes                        = const - spikes

    spikes                        = spikes.reshape(-1)
    tempzero                      = torch.zeros(spikes.shape)
    spikes[spikes >= tempzero]    = 1
    spikes[spikes <  tempzero]    = 0
    spikes                        = spikes.reshape(time,temprows,tempcols,tempnprev)

    return spikes

In [ ]:
"""
 Author: Shanmuga Venkatachalam
"""

### Enabling CUDA support for GPU ###
cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

### Training Parameter Definitions ###
inc_learn = 1
breakpoint1 = 60000
interval1   = 1000
breakpoint2 = 10000
interval2   = 1000

start_pos = 0
size_offset = 28

### Model Parameter Definitions ###
capture = 1
search  = 0
backoff = 0.625
frac = 0
wBase = 12
wMax = 16

imageSize = size_offset
kernelSize = 5
rfSize = 5
stride = 2
dilation = 1
numDistalInputs = (rfSize ** 2) * 2
numSegments = 36
numDendrites = 10
numNeurons = int((math.floor((imageSize - kernelSize) / stride) + 1) ** 2)
threshold = 0
voting_thresh = 340

In [ ]:
### Active Dendrite Layer ###

    # Implements a single NeuTNN layer with multiple neurons, dendrites, and segments.
    # Segments are assumed to be distal, while the proximal input is simply used as enable during weight update.
    # Capable of online continuous learning via STDP/R-STDP.

    # Args: numDistalInputs - Number of distal inputs to a single neuron. Within a neuron, the inputs are shared across all segments.
    #       numSegments     - Number of segments per dendrite per neuron.
    #       numDendrites    - Number of dendrites per neuron.
    #       numNeurons      - Number of neurons in the layer.
    #       threshold       - Excitation threshold for neuron. An output spike is generated when neuron's body potential reaches
    #                         this threshold.
    #       wMax            - Maximum value for synaptic weights.
    #       wInit           - Type of initialization for synaptic weights. Supports 'zero', 'uniform', and 'normal' initializations.
    #       rewardEn        - Enable reinforcement for STDP (R-STDP).
    #       device          - Device type (e.g. cpu, cuda).
    #
    # Input: data           - Input data to the layer of shape [numNeurons, numDistalInputs].
    #
    # __call__ is the forward function whereas "update" performs STDP update in either unsupervised or supervised fashion.
class activeDendriteLayer(nn.Module):
    def __init__(self,
                 numDistalInputs: int,
                 numSegments: int,
                 numDendrites: int,
                 numNeurons: int,
                 threshold: int,
                 wMax: int = 8,
                 wInit: str = "uniform",
                 rewardEn: bool = True,
                 device="cpu"
                ):

        super(activeDendriteLayer, self).__init__()

        self.numDistalInputs = numDistalInputs
        self.numSegments = numSegments
        self.numDendrites = numDendrites
        self.numNeurons = numNeurons
        self.threshold = threshold
        self.wMax = wMax
        self.wInit = wInit
        self.rewardEn = rewardEn
        self.device = device

        self.totalSegments = self.numNeurons*self.numDendrites*self.numSegments

        if self.wInit       == "zero":
            self.weights   = nn.Parameter(torch.zeros(self.totalSegments,
                                                        self.numDistalInputs), requires_grad=False)

        elif self.wInit     == "uniform":
            self.weights   = nn.Parameter(torch.randint(low = 0, high = self.wMax + 1,
                                            size=(self.totalSegments,
                                                self.numDistalInputs)).type(torch.FloatTensor),
                                            requires_grad=False)

        elif self.wInit     == "normal":
            self.weights   = nn.Parameter(torch.round(( (self.wMax+1)/2 + \
                                                        torch.randn(self.totalSegments, self.numDistalInputs))\
                                                        .clamp_(0,self.wMax)), requires_grad=False)


    ########################################### Inference function ###########################################
    def __call__(self, data):
        # Data is of shape [self.numNeurons, self.numDistalInputs]

        inputData = data.unsqueeze(1).repeat(1,self.numDendrites*self.numSegments,1)\
                        .reshape(self.totalSegments,self.numDistalInputs)

        # Applying weights
        weightedInput = inputData * self.weights

        # Weighted input summation - SNL
        summedOutput = torch.sum(weightedInput, dim=1)

        zeroCondition = summedOutput < self.threshold
        summedOutput[zeroCondition] = 0

        # Winner Take All inhibition across segments
        reshapedSummedOutput = summedOutput.reshape(-1,self.numSegments)

        self.wtaOutput = torch.zeros(self.numNeurons*self.numDendrites, self.numSegments,
                                     dtype=reshapedSummedOutput.dtype, device=self.device)

        maxval, indices = torch.max(reshapedSummedOutput, dim=1)

        self.wtaOutput.scatter_(1, indices.unsqueeze(1), maxval.unsqueeze(1))

        outputData = (self.wtaOutput>0).reshape(-1).unsqueeze(1).repeat(1,self.numDistalInputs)

        return inputData, outputData


    ########################################### Update function ###########################################
    def update(self, inputData, inputLabels, outputData, weights, capture, search, backoff, wBase, rewardEn, frac=0.25):

        if rewardEn:
            captureVal = (inputData == 1) * (outputData == 1) * (inputLabels == 1) * capture + (inputData == 0) * (outputData == 1) * (inputLabels == 0) * capture * frac
            backoffVal = (inputData == 0) * (outputData == 1) * (inputLabels == 1) * backoff + (inputData == 1) * (outputData == 1) * (inputLabels == 0) * backoff * frac
            searchCondition  = (inputData == 1) * (outputData == 0) * (inputLabels == 1)
        else:
            captureVal = (inputData == 1) * (outputData == 1) * capture
            backoffVal = (inputData == 0) * (outputData == 1) * backoff
            searchCondition  = (inputData == 1) * (outputData == 0)

        # Capture and Backoff
        weights += captureVal - backoffVal

        # Search
        weights[searchCondition] = torch.maximum(weights[searchCondition],
                                                 torch.minimum(weights[searchCondition] + search,
                                                               torch.tensor([wBase], device=self.device)))

        return weights.clamp_(0, self.wMax)


### MNIST dataset loading and preprocessing ###

train_loader = DataLoader(MNIST('./data', True, download=True, transform=transforms.Compose(
          [
            transforms.ToTensor(),
            PosNeg_SpikeEvents(0.7),
            Crop(start_pos, size_offset)
          ]
                                                                                       )
      ),
batch_size=1,
shuffle=False
)

test_loader = DataLoader(MNIST('./data', False, download=True, transform=transforms.Compose(
            [
              transforms.ToTensor(),
              PosNeg_SpikeEvents(0.7),
              Crop(start_pos, size_offset)
            ]
                                                                                       )
      ),
batch_size=1,
shuffle=False
)

def train_func(numDistalInputs, numSegments, numDendrites, numNeurons, threshold, capture, search, backoff, frac, wBase, device, rfSize, wMax, voting_thresh=0, stride=1, dilation=2):

    myMNISTNetwork = activeDendriteLayer(numDistalInputs = numDistalInputs,
                            numSegments = numSegments,
                            numDendrites = numDendrites,
                            numNeurons = numNeurons,
                            threshold = threshold,
                            wMax=wMax,
                            device = device
                            )

    if cuda:
        myMNISTNetwork.cuda()

    unfold = nn.Unfold(kernel_size = (rfSize, rfSize), dilation = dilation, stride = stride)

    for epochs in range(1):
        start = time.time()
        error1 = 0
        error2 = 0
        errorlist1 = []
        errorlist2 = []

        for idx, (data,target) in enumerate(train_loader):

            if cuda:
                data                    = data.cuda()
                target                  = target.cuda()

            if(idx == 0):
              print("data peak:\n", data.shape)
            ### Uncomment to demonstrate online learning
            """
            if (idx+1) > 30000:
                data                    = torch.transpose(data,2,3)
            """

            label = target[0]

            ### Uncomment to demonstrate online learning
            """
            if (idx+1) > 50000:
                if (target[0]%2) == 0:
                    label = target[0] + 1
                else:
                    label = target[0] - 1
            """

            sliced_data = unfold(data).transpose(1,2).squeeze()
            if(idx == 0):
              print("unfold peak:\n", unfold)
              print("sliced_data peak:\n", sliced_data.shape)

            inputData, outputData = myMNISTNetwork(sliced_data)

            labelCount = torch.max(myMNISTNetwork.wtaOutput, dim=1)[0] > voting_thresh
            labelCount = labelCount.reshape(-1,10)
            labelVotes = torch.sum(labelCount, dim=0)

            if torch.argmax(labelVotes) != label:
                error1 += 1
                error2 += 1

            inputLabels = F.one_hot(torch.tensor([label], device=device),num_classes=10).squeeze()\
                                    .unsqueeze(0).repeat(myMNISTNetwork.numNeurons,1)\
                                    .unsqueeze(2).repeat(1,1,myMNISTNetwork.numSegments * myMNISTNetwork.numDistalInputs)\
                                    .reshape(myMNISTNetwork.totalSegments,myMNISTNetwork.numDistalInputs)

            myMNISTNetwork.weights = myMNISTNetwork.update(inputData, inputLabels, outputData, myMNISTNetwork.weights,\
                                                            capture, search, backoff, wBase, myMNISTNetwork.rewardEn, frac)

            if (idx+1)%interval1 == 0:
                errorlist1.append(error1/(idx+1))
                errorlist2.append(error2/interval1)
                error2 = 0

            endt = time.time()

        end = time.time()
        training_acc = (breakpoint1-error1)*100/breakpoint1
        print("Training for {0} samples done in {1}".format(idx, end-start))
        print("Training Accuracy for {1} epochs: {0}%".format(training_acc, epochs+1))

    error3 = 0
    start = time.time()

    for idx, (data,target) in enumerate(test_loader):

        if cuda:
            data                    = data.cuda()
            target                  = target.cuda()

        # data                    = torch.transpose(data,2,3) # Uncomment to demonstrate online learning

        label = target[0]

        ### Uncomment to demonstrate online learning
        """
        if (target[0]%2) == 0:
            label = target[0] + 1
        else:
            label = target[0] - 1
        """

        sliced_data = unfold(data).transpose(1,2).squeeze()

        inputData, outputData = myMNISTNetwork(sliced_data)

        labelCount = torch.max(myMNISTNetwork.wtaOutput, dim=1)[0] > voting_thresh
        labelCount = labelCount.reshape(-1,10)
        labelVotes = torch.sum(labelCount, dim=0)

        if torch.argmax(labelVotes) != label:
            error1 += 1
            error2 += 1
            error3 += 1

        if inc_learn == 1:
            inputLabels = F.one_hot(torch.tensor([label], device=device),num_classes=10).squeeze()\
                                    .unsqueeze(0).repeat(myMNISTNetwork.numNeurons,1)\
                                    .unsqueeze(2).repeat(1,1,myMNISTNetwork.numSegments * myMNISTNetwork.numDistalInputs)\
                                    .reshape(myMNISTNetwork.totalSegments,myMNISTNetwork.numDistalInputs)

            myMNISTNetwork.weights = myMNISTNetwork.update(inputData, inputLabels, outputData, myMNISTNetwork.weights,\
                                                            capture, search, backoff, wBase, myMNISTNetwork.rewardEn, frac)

        if (idx+1)%interval2 == 0:
            errorlist1.append(error1/(idx+1+breakpoint1))
            errorlist2.append(error2/interval2)
            error2 = 0

        endt = time.time()

    end = time.time()
    testing_acc = (breakpoint2-error3)*100/breakpoint2
    print("Testing for {0} samples done in {1}".format(idx, end-start))
    print("Testing Accuracy: {0}%".format(testing_acc))

    return training_acc, testing_acc, errorlist2, myMNISTNetwork

training_accuracy, testing_accuracy, errorlist2, myMNISTNetwork = train_func(numDistalInputs = numDistalInputs,
                                                        numSegments = numSegments,
                                                        numDendrites = numDendrites,
                                                        numNeurons = numNeurons,
                                                        threshold = threshold,
                                                        capture = capture,
                                                        search = search,
                                                        backoff = backoff,
                                                        frac = frac,
                                                        wBase = wBase,
                                                        device = device,
                                                        rfSize = rfSize,
                                                        wMax = wMax,
                                                        voting_thresh = voting_thresh,
                                                        stride = stride,
                                                        dilation=dilation
                                                        )

In [ ]:
### Storing image ###

plt.figure(figsize=(10,4))

dx = [i for i in range(1,int(breakpoint1/interval1+breakpoint2/interval2+1))]
plt.plot(dx, errorlist2, color='red', linestyle='solid', linewidth=1.5)

plt.xlabel("Samples (x1000)")
plt.ylabel("Error Rate")
plt.xticks(np.arange(1,int(breakpoint1/interval1+breakpoint2/interval2+1),2))
plt.legend(["activeDendriteLayer"])
plt.savefig("online_adaptivity.png")